<a href="https://colab.research.google.com/github/gph05010/Daily-Study-Log-DeepLearning/blob/main/DeepLearning/ex11_Transformers(%EB%89%B4%EC%8A%A4%EC%B9%B4%ED%85%8C%EA%B3%A0%EB%A6%AC_%EB%B6%84%EB%A5%98_%EC%8B%A4%EC%8A%B5).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### 학습목표
- HuggingFace에 업로드되어 있는 모델을 Fine-Tunning하여 나만의 모델로 만들기를 할 수 있다.
- 모델 학습 후 HuggingFace에 업로드할 수 있다.
- HuggingFace에 업로드된 나의 모델을 불러 예측할 수 있다.

#### Fine-Tunning(미세조정)이란?
- 사전학습된 대규모 언어모델(LLM)을 특정 작업이나 특정 도메인에 맞게 미세조정하는 기법
- LLM이 특정 작업에서 성능을 최적화하기 위해 추가학습

### 뉴스카테고리 분류(다중분류 실습)
  - KLUE/YNAT 데이터셋 사용
  - datasets에서 제공하는 뉴스 데이터셋

In [ ]:
# 연합뉴스 데이터셋 불러오기~
from datasets import load_dataset
klue_train = load_dataset('klue', 'ynat', split = 'train')
klue_eval = load_dataset('klue', 'ynat', split = 'validation')

In [ ]:
# 데이터 확인
klue_train

In [ ]:
# 1개 데이터 확인
klue_train[0]
# guid : 데이터 식별자
# title : 뉴스 제목
# label : 정답 데이터
# url : 뉴스 url
# date : 날짜

- 뉴스 제목을 활용하여 카테고리 분류 실습

In [ ]:
# 실습에 사용하지 않는 불필요한 컬럼 제거
klue_train = klue_train.remove_columns(['guid', 'url', 'date'])
klue_eval = klue_eval.remove_columns(['guid', 'url', 'date'])

In [ ]:
klue_train

In [ ]:
# 정답 데이터의 class 확인
klue_label = klue_train.features['label'] # 총 7개의 class

In [ ]:
klue_train[0]

✅ 카테고리 (label)

| 숫자 라벨 | 문자형 라벨 |
| ----- | ------ |
| 0     | IT과학   |
| 1     | 경제     |
| 2     | 사회     |
| 3     | 생활문화   |
| 4     | 세계     |
| 5     | 스포츠    |
| 6     | 정치     |



- 우리가 보기 쉽도록 'label_str' 컬럼을 추가

In [ ]:
# 숫자로 된 label 값을 해당하는 문자열로 변환하여 새로운 컬럼 생성
def make_str_label (data) :
  data['label_str'] = klue_label.int2str(data['label'])
  return data

In [ ]:
klue_train = klue_train.map(make_str_label, batched=True, batch_size=1000)
klue_eval = klue_eval.map(make_str_label, batched=True, batch_size=1000)

In [ ]:
klue_train[2]

In [ ]:
klue_eval

In [ ]:
from sklearn.model_selection import train_test_split

klue_train.train_test_split(
    test_size = 10000,
    shuffle = True,
    seed = 4
)

In [ ]:
# 학습용/검증용/테스트용 데이터셋으로 분리
train_dataset = klue_train.train_test_split(test_size = 10000, shuffle = True, seed = 4)['train']
val_dataset = klue_train.train_test_split(test_size = 10000, shuffle = True, seed = 4)['test']

In [ ]:
train_dataset

In [ ]:
val_dataset

### 모델학습
##### RoBERTa : facebook AI가 2019년도에 발표한 Bert 모델의 성능을 개선한 모델

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
# 라이브러리 불러오기
checkpoint = 'klue/roberta-base'

# 토큰화 도구(tokenizer) 생성하기 -> AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained(checkpoint)
# 모델 객체(mod el) 생성하기 -> AutoModelForSequenceClassification
model = AutoModelForSequenceClassification.from_pretrained(checkpoint, num_labels = 7)

In [ ]:
# 클래스의 개수 확인하는 방법
len(klue_label.names)

In [ ]:
# 토큰화 함수 생성(tokenizer_function) -> 패딩 : 최대 길이, 길면 잘라주세요
def tokenizer_function (example) :
  return tokenizer(example['title'],     # 데이터셋
                   max_length = 128,        # 문장의 최대 길이
                   padding = 'max_length',  # 최대 길이보다 짧을 경우 -> 0값으로 패딩
                   truncation = True)

# train, val, eval 데이터셋 모두 토큰화 진행
# train_dataset, valid_dataset, test_dataset
train_dataset = train_dataset.map(tokenizer_function, batched = True)
valid_dataset = val_dataset.map(tokenizer_function, batched = True)
test_dataset = klue_eval.map(tokenizer_function, batched = True)

In [ ]:
print(train_dataset[0])

In [ ]:
# 모델의 정확도 판단을 위하여 정확도 계산 함수
import numpy as np
# 모델 출력 결과를 통해 예측값 확인 -> 실제 값(label)과 비교를 통하여 정확도 계산
def compute_metrics(eval_pred) :
  # eval_pred : Huggingface에서 Trainer 객체를 사용할 때 자동으로 전달되는 값 -> 튜플(logits, labels). logits : 확률값
  logits, labels = eval_pred
  pred = np.argmax(logits, axis = -1) # 모델이 출력한 확률값 중 최댓값의 인덱스를 출력

  # 정확도 계산 -> pred == labels -> True(1), False(0)
  return {'accuracy' : (pred == labels).mean()}

- 학습 모델 설정

  - Huggingface의 Transformers 라이브러리의 TrainingArguments()를 사용
  - 모델 학습 과정에 필요한 다양한 설정값을 정의
  - 각 매개변수는 학습 성능과 효율정을 조절

  
  ### ✅ Huggingface `TrainingArguments` 설정 요약

| 매개변수                  | 설명 |
|---------------------------|------|
| `output_dir="./results"` | 학습 결과 저장 디렉토리 (모델, 로그, 체크포인트 등 저장) |
| `num_train_epochs=10`    | 전체 학습 데이터 반복 횟수 (Epoch 수) |
| `per_device_train_batch_size=4` | 각 디바이스(GPU/CPU)별 배치 크기 |
| `gradient_accumulation_steps=1` | 그래디언트 누적 스텝 수 (메모리 절약용) |
| `optim="paged_adamw_32bit"` | 32비트 정밀도 사용 AdamW 옵티마이저 변형 |
| `save_steps=25`          | 몇 스텝마다 모델을 저장할지 설정 |
| `logging_steps=25`       | 몇 스텝마다 로그를 기록할지 설정 |
| `learning_rate=2e-4`     | 학습률 설정 |
| `weight_decay=0.001`     | 가중치 감소 계수 (정규화 효과로 과적합 방지) |
| `fp16=False`             | 16비트 부동소수점 정밀도 사용 여부 |
| `bf16=False`             | BF16 정밀도 사용 여부 (Brain Floating Point) |
| `max_grad_norm=0.3`      | 그래디언트 최대 노름(norm) 값 설정 (폭발 방지) |
| `max_steps=-1`           | 전체 학습 스텝 수 (-1은 전체 epoch만큼 학습) |
| `warmup_ratio=0.03`      | 학습률 워밍업 비율 (초기 안정적 학습 유도) |
| `group_by_length=True`   | 입력 시퀀스 길이에 따라 배치 그룹화 (패딩 최적화) |
| `lr_scheduler_type="constant"` | 학습률 스케줄러 유형 (constant는 고정) |
| `report_to="tensorboard"` | TensorBoard에 로그 기록 여부 설정 |


In [ ]:
%cd /content/drive/MyDrive/00 딥러닝

In [ ]:
# 모델 하이퍼파라미터 설정 및 트레이닝
from transformers import Trainer, TrainingArguments

training_args = TrainingArguments(
    output_dir="./results/roberta-base-klue-ynat-classification",
    num_train_epochs=1,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    learning_rate=5e-5,
    push_to_hub=False
)

# 학습 객체
trainer = Trainer(model = model,
                  args = training_args,
                  train_dataset = train_dataset,
                  eval_dataset = valid_dataset,
                  compute_metrics = compute_metrics)

# 모델 학습
trainer.train()

In [ ]:
# token 파일로 저장해두기
# open 사용해서 불러 사용할 수 있게 됨

# 폴더 생성 -> api_key 저장
import os # 파일, 폴더 관리 시스템
if not os.path.exists('./key'):
  os.mkdir('./key')

# 나의 api_key 등록
api_key = '키'

with open('./key/huggingface_api_key', 'w') as f:
  f.write(api_key)

In [ ]:
# 허깅페이스 로그인
from huggingface_hub import login

# 파일형태의 api_key 불러오기
with open('./key/huggingface_api_key', 'r') as f:
  api_key = f.read().strip()

login(token = api_key)

In [ ]:
# 허깅페이스에 업로드하기
# 닉네임/사용한 사전 모델, 데이터, task
report_id = f"닉네임/roberta-base-klue-ynat-classification"

# trainer, model, tokenizer
trainer.save_model(report_id)
model.save_pretrained(report_id)
tokenizer.save_pretrained(report_id)

trainer.push_to_hub(report_id)

In [ ]:
# 업로드된 나의 모델 불러와서 사용하기
# pipeline 사용하기
from transformers import pipeline

my_model = pipeline(task = 'text-classification', model = '닉네임/roberta-base-klue-ynat-classification')